[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-03-pipeline-api.ipynb#scrollTo=aa110011)

---
# Day 3 · The Pipeline API — Classification, NER, Summarization, and More
**certified-journeys / huggingface-nlp-certified** · Day 3 · Core Concepts

> **Goal for today:** Use five different pipeline tasks (zero-shot classification, NER, text generation, summarization, and benchmarking), understand when to use each, and measure real inference throughput.


## Step 1 · Install Dependencies


In [ ]:
%pip install -q transformers accelerate


## Step 2 · The Pipeline API — Task Overview

The `pipeline()` function is the highest-level abstraction in `transformers`. It supports ~30 tasks out of the box:

| Task string | What it does | Typical model family |
|---|---|---|
| `"sentiment-analysis"` | Binary or multi-class sentiment | DistilBERT, RoBERTa |
| `"zero-shot-classification"` | Classify without task-specific training | BART, DeBERTa |
| `"ner"` | Named entity recognition (spans + types) | BERT, RoBERTa |
| `"text-generation"` | Auto-regressive text generation | GPT-2, LLaMA |
| `"summarization"` | Extractive/abstractive summarisation | BART, T5, Pegasus |
| `"translation"` | Neural machine translation | Helsinki-NLP Opus |
| `"question-answering"` | Extractive QA from context | BERT, RoBERTa |
| `"fill-mask"` | Predict masked tokens (MLM) | BERT, RoBERTa |

Each pipeline handles: tokenization → batching → model forward pass → post-processing. You only see clean Python objects.


## Step 3 · Zero-Shot Classification

Zero-shot classification lets you classify text into arbitrary categories *you define at inference time* — no fine-tuning required. Under the hood, it uses a Natural Language Inference (NLI) model to score whether the text *entails* each candidate label.

**How it works:**
1. For each candidate label, the model checks: "Does this text entail 'This example is about [LABEL]'?"
2. The entailment scores are normalised to probabilities.
3. `multi_label=True` allows multiple labels above the threshold simultaneously.


In [ ]:
from transformers import pipeline

# facebook/bart-large-mnli is the standard zero-shot model
zs_clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

text = (
    "The Federal Reserve raised interest rates by 25 basis points, citing "
    "persistent inflation above the 2% target. Markets reacted with a "
    "brief sell-off before recovering by close of trading."
)

candidate_labels = ["economics", "politics", "sports", "technology", "climate"]

result = zs_clf(text, candidate_labels=candidate_labels)

print("=== Zero-Shot Classification ===")
print(f"Text: {text[:80]}...\n")
print(f"{'Label':<15} {'Score':>8}")
print("-" * 25)
for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 30)
    print(f"{label:<15} {score:>7.3f}  {bar}")

# Multi-label: allow more than one label to "win"
print("\n=== Multi-label (threshold not applied — all scores shown) ===")
multi_result = zs_clf(
    text,
    candidate_labels=candidate_labels,
    multi_label=True,  # scores are independent sigmoid probs, not softmax
)
for label, score in zip(multi_result["labels"], multi_result["scores"]):
    print(f"  {label:<15} {score:.3f}")


### What just happened?

- **Single-label mode** applies softmax across all candidates — scores sum to 1.0.
- **Multi-label mode** applies sigmoid independently — each score is a probability of that label being relevant, and they do NOT sum to 1.0.
- The model never saw the candidate labels during training; it generalises via NLI entailment reasoning.
- **Key insight:** Zero-shot classification is ideal for rapid prototyping and low-data scenarios. It underperforms fine-tuned classifiers when training data is available, but saves weeks of labelling.


## Step 4 · Named Entity Recognition (NER)

NER identifies spans of text that refer to named entities (people, organisations, locations, dates, …) and assigns each span a type label.

**Key concepts:**
- **B-/I- prefixes** (BIO tagging): `B-PER` starts a person entity; `I-PER` continues it.
- `aggregation_strategy="simple"` merges consecutive BIO tokens into a single span object.
- The pipeline returns: `entity_group`, `score`, `word`, `start`, `end` (character offsets).


In [ ]:
from transformers import pipeline

# dslim/bert-base-NER is a widely-used, small NER model
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",  # merge B-/I- tokens into spans
)

text = (
    "Elon Musk, CEO of Tesla and SpaceX, announced a new Starship launch "
    "from Boca Chica, Texas on Monday. The European Space Agency expressed "
    "interest in a collaboration."
)

entities = ner(text)

print(f"Text: {text}\n")
print(f"{'Entity':<30} {'Type':<8} {'Score':>7}  {'Start':>5}  {'End':>4}")
print("-" * 65)
for ent in entities:
    print(
        f"{ent['word']:<30} "
        f"{ent['entity_group']:<8} "
        f"{ent['score']:>7.3f}  "
        f"{ent['start']:>5}  "
        f"{ent['end']:>4}"
    )

# Verify character offsets by slicing the original text
print("\nVerifying character offsets:")
for ent in entities[:3]:
    span = text[ent["start"]:ent["end"]]
    print(f"  [{ent['start']}:{ent['end']}] = '{span}' (expected: '{ent['word']}')")


### What just happened?

- **`aggregation_strategy="simple"`** collapses `['Elo', '##n', 'Musk']` into a single `{word: 'Elon Musk', entity_group: 'PER'}` object.
- **Character offsets** (`start`, `end`) point into the *original* string — you can use them to highlight entities in a UI without re-tokenizing.
- **Confidence scores** below ~0.85 are often false positives; add a threshold filter in production.
- **Key insight:** Always verify entity boundaries with character offsets. Tokenizer subword splits can sometimes shift boundaries by one character, especially around punctuation.


## Step 5 · Text Generation with GPT-2

Text generation pipelines use auto-regressive models — they predict the next token one at a time, conditioning on all previous tokens. Key parameters:

| Parameter | Role |
|---|---|
| `max_new_tokens` | Number of tokens to generate (not counting the prompt) |
| `num_return_sequences` | Generate multiple independent completions |
| `do_sample` | `True` = sampling (creative); `False` = greedy/beam |
| `temperature` | Sharpness of sampling distribution; < 1 = more focused |
| `top_p` | Nucleus sampling: sample from top-p probability mass |
| `pad_token_id` | Required for batched generation; set to `eos_token_id` for GPT-2 |


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2",
    # GPT-2 has no pad token by default; setting it prevents a warning
    pad_token_id=50256,  # 50256 = EOS token for GPT-2
)

prompt = "The future of natural language processing is"

outputs = generator(
    prompt,
    max_new_tokens=100,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.85,       # slightly below 1 for more coherent output
    top_p=0.92,             # nucleus sampling
    truncation=True,
)

print(f"Prompt: '{prompt}'\n")
for i, output in enumerate(outputs):
    generated_text = output["generated_text"]
    # Strip the prompt from the start so we only see the new content
    continuation = generated_text[len(prompt):].strip()
    print(f"--- Completion {i+1} ---")
    print(continuation)
    print()


### What just happened?

- `num_return_sequences=3` runs three independent forward passes (or a batched beam search) — each output is different because of stochastic sampling.
- **`max_new_tokens` vs `max_length`**: `max_new_tokens` counts only the generated tokens; `max_length` counts the prompt + generated tokens. Prefer `max_new_tokens` to avoid silently truncating long prompts.
- `temperature=0.85` divides the logits before softmax, making the distribution sharper (less random) than `temperature=1.0`.
- **Key insight:** For deterministic, reproducible generation (testing, evaluation), set `do_sample=False` and `num_beams=1` (greedy). For creative tasks, `do_sample=True` with `top_p=0.9` is a sensible default.


## Step 6 · Summarization — Controlling Output Length

Abstractive summarization generates a new summary (not just extracting sentences). The two key length parameters constrain the model's output:

- `min_length` — generation will not stop before this many tokens
- `max_length` — generation will be cut off here (hard cap)

We use `facebook/bart-large-cnn`, fine-tuned on CNN/DailyMail news articles.


In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
)

# A ~500-word article excerpt about large language models
article = """
Large language models (LLMs) have fundamentally changed how software engineers 
build natural language processing systems. Rather than designing task-specific 
pipelines with hand-crafted feature engineering, teams now fine-tune or prompt 
a single pre-trained model to handle dozens of different tasks — from sentiment 
analysis and entity extraction to code generation and question answering.

The shift began in earnest with the publication of BERT in 2018, which demonstrated 
that bidirectional pre-training on large unlabelled corpora could produce 
representations that transfer remarkably well to downstream tasks with minimal 
task-specific data. BERT's successors — RoBERTa, ALBERT, DeBERTa — pushed 
accuracy higher on virtually every NLP benchmark.

On the generation side, the GPT series showed that scaling autoregressive 
language models produced qualitative jumps in capability. GPT-3, with 175 
billion parameters, could perform few-shot learning — solving new tasks from 
just a few examples in the prompt — without any gradient updates. This sparked 
widespread interest in prompt engineering as a discipline.

Today, engineers working with LLMs must understand a growing toolkit: 
retrieval-augmented generation (RAG) to ground models in up-to-date factual 
knowledge; fine-tuning techniques like LoRA and QLoRA that make large-model 
adaptation accessible on a single GPU; and evaluation frameworks that go beyond 
accuracy to measure factuality, calibration, and safety.

The Hugging Face ecosystem has emerged as the practical standard for this work. 
The Hub hosts tens of thousands of models and datasets; the transformers, 
datasets, and evaluate libraries provide consistent APIs across research and 
production; and the PEFT library offers ready-to-use parameter-efficient 
fine-tuning methods. For engineers entering NLP in 2024, fluency in this 
ecosystem is as foundational as knowing how to write a SQL query.
""".strip()

print(f"Original word count: {len(article.split())} words\n")

# Compare short vs long summaries
for min_l, max_l in [(30, 60), (60, 130)]:
    summary = summarizer(article, min_length=min_l, max_length=max_l)[0]["summary_text"]
    print(f"--- min_length={min_l}, max_length={max_l} ---")
    print(f"Words: {len(summary.split())}")
    print(summary)
    print()


### What just happened?

- **`min_length=30`** forces the model to generate at least 30 tokens before emitting EOS — prevents a one-sentence summary when you need more coverage.
- **`max_length=130`** imposes a hard upper bound; the model may stop earlier if it generates EOS naturally.
- BART-CNN is trained on news; it performs best on formal, well-structured prose. It may produce repetition or hallucinations on informal text.
- **Key insight:** Summarization quality degrades when the input is longer than the model's max input length (1024 tokens for BART). For very long documents, split into chunks, summarize each, then summarize the summaries (map-reduce pattern).


## Step 7 · Benchmarking Pipeline Inference Time

Before optimising, measure. Use `time.perf_counter` (nanosecond resolution, wall-clock time) to benchmark batch inference.

**What to benchmark:**
- Single-example latency (P50, P99)
- Batch throughput (examples/second)
- Effect of batch size on GPU utilisation (if GPU available)

On CPU, batching with `batch_size=8` or higher is the next-best speedup after moving to GPU (`device=0`).


In [ ]:
import time
from transformers import pipeline

# Use a small, fast model for the benchmark
clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

# Generate 32 synthetic examples
N = 32
examples = [
    f"Example sentence number {i} for benchmarking pipeline inference speed."
    for i in range(N)
]

def benchmark(examples, batch_size=1, n_runs=3):
    """Run the pipeline n_runs times and return mean throughput (examples/sec)."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = clf(examples, batch_size=batch_size)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    mean_secs = sum(times) / len(times)
    throughput = len(examples) / mean_secs
    return mean_secs, throughput

print(f"Benchmarking {N} examples, averaged over 3 runs each\n")
print(f"{'batch_size':>12}  {'mean latency':>14}  {'throughput':>16}")
print("-" * 48)

for bs in [1, 4, 8, 16, 32]:
    secs, tps = benchmark(examples, batch_size=bs)
    print(f"{bs:>12}  {secs:>12.3f}s  {tps:>14.1f} ex/s")

print("\nNote: On CPU, larger batch_size amortises Python overhead.")
print("On GPU (device=0), batch_size=32+ is typically optimal.")


### What just happened?

- **Batch size 1** is the slowest: each call has fixed Python overhead (function call, tensor allocation) that dominates when the compute is tiny.
- **Larger batches** amortise overhead — throughput increases until the batch fills available memory or compute.
- `time.perf_counter()` is preferred over `time.time()` for short intervals because it uses a high-resolution monotonic clock not affected by wall-clock adjustments.
- **Key insight:** Always warm up the model with one dummy batch before benchmarking — the first call loads weights and JIT-compiles kernels, which distorts timing significantly.


## Challenge

Build a mini multi-task inference function that accepts a piece of text and returns: (1) sentiment label + score, (2) all named entities, and (3) a 30–60 token summary. Benchmark it on a batch of 8 texts and report per-task latency.


In [ ]:
# Challenge: Multi-task pipeline runner
# Your solution here

import time
from transformers import pipeline

# Step 1: Instantiate pipelines for sentiment, NER, and summarization
# sentiment_pipe = pipeline("sentiment-analysis", model="...")
# ner_pipe       = pipeline("ner", model="...", aggregation_strategy="simple")
# summarizer     = pipeline("summarization", model="...")

# Step 2: Write a function analyse(texts: list[str]) -> list[dict]
# Each output dict should have keys: text, sentiment, entities, summary
# def analyse(texts):
#     results = []
#     ...
#     return results

# Step 3: Create 8 test sentences, call analyse(), and print results
texts = [
    "Apple announced record quarterly earnings in Cupertino, California.",
    "The climate summit in Paris failed to reach a binding agreement.",
    "Scientists at MIT developed a new battery that charges in 5 minutes.",
    "Liverpool defeated Manchester United 3-0 at Anfield on Saturday.",
    "The Federal Reserve kept interest rates unchanged at its March meeting.",
    "SpaceX successfully landed its Starship prototype for the first time.",
    "Amazon Web Services reported a 17% revenue increase year over year.",
    "WHO declared the end of the mpox public health emergency.",
]

# Step 4: Benchmark each pipeline separately using time.perf_counter
# Report: task name | total time | throughput (examples/sec)


---
## Day 3 Key Concepts Recap

| Concept | What to remember |
|---|---|
| `zero-shot-classification` | NLI-based; no fine-tuning needed; `multi_label=True` for independent scores |
| NER `aggregation_strategy` | `"simple"` merges BIO tokens into span objects with char offsets |
| `max_new_tokens` | Counts only generated tokens, not the prompt — prefer over `max_length` |
| `do_sample + temperature` | `temperature < 1` = sharper distribution; `do_sample=False` = greedy |
| Summarization length | `min_length` prevents premature EOS; `max_length` is a hard cap |
| Batch size benchmarking | Warm up first; CPU benefits from larger batches to amortise overhead |
| GPU speedup | `device=0` in pipeline() moves inference to GPU; biggest single speedup |

> **Tip:** Set device=0 in the pipeline() call to move inference to GPU — on CPU, batching with batch_size=8 or larger is the next-best speedup.

---
## What's next
**Day 4** → Fine-tuning BERT on a custom classification task — you will train a model end-to-end using the `Trainer` API.

Mark Day 3 complete in your [tracker](../index.html).
